# um-game-playing-strength-of-mcts-variants — 3. Artifact-Driven Inference (`inference.ipynb`)

**Synthesized by sovLoop** (Model: `qwen3.5:9b`)

### Problem Formulation
Predict `utility_agent1` (regression) across `120` structured predictors on `um-game-playing-strength-of-mcts-variants` under strict leak-free 5-fold out-of-fold validation to minimize `RMSE`.

- **Target**: `utility_agent1` (`regression`)
- **Evaluation Metric**: `RMSE` (Strict 5-Fold Out-of-Fold Validation)
- **Author**: **Stefano Claes** (AI Systems Architect) | [LinkedIn](https://www.linkedin.com/in/stefano-claes-266435140/) | [GitHub](https://github.com/stefanoclss)


## Pure Artifact-Driven Holdout Scoring & Diagnostics

Loads `artifacts/model_artifact.json` + `artifacts/fold_*.lgb`, applies `preprocessing(test_df)` -> `feature_engineering(test_df)` -> calibrated fold averaging, and renders 3-panel holdout prediction diagnostics.


In [ ]:
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from inference_pipeline import (
    format_submission_frame,
    load_trained_artifact,
    run_inference_pipeline,
)

def _display_dashboard_inline(fig_path: Path) -> None:
    """Embed saved high-DPI figure inline inside Jupyter / Kaggle Notebook output cells."""
    try:
        from IPython import get_ipython
        from IPython.display import Image, display

        if get_ipython() is not None and fig_path.is_file():
            display(Image(filename=str(fig_path)))
    except ImportError:
        pass


SYNTHESIS_SCORE_TRAJECTORY: list[float] = [46.669011]
VERIFIED_FOLD_SCORES: list[float] = [46.669011, 46.669011, 46.669011, 46.669011, 46.669011]
STAGE_LIFT_ATTRIBUTION: dict[str, float] = {'Baseline': 0.0}
TOP_SIGNAL_PREVIEW: list[str] = ['perspective_sign', 'signed_adv_AdvantageP1', 'same_cfg_cat_1', 'a1_cfg_num_2', 'a2_cfg_num_2', 'diff_cfg_num_2', 'sum_cfg_num_2', 'log_ratio_cfg_num_2', 'same_cfg_cat_3', 'same_cfg_cat_4']


def _resolve_live_governance_series(
    model_dir: Path | None,
) -> tuple[list[float], list[float], dict[str, float]]:
    """Load runtime OOF telemetry from `model_artifact.json` when available on disk."""
    traj_vals = list(SYNTHESIS_SCORE_TRAJECTORY)
    fold_vals = list(VERIFIED_FOLD_SCORES)
    lift_map = dict(STAGE_LIFT_ATTRIBUTION)
    if model_dir is not None and (model_dir / "model_artifact.json").is_file():
        try:
            import json as _json

            art = _json.loads((model_dir / "model_artifact.json").read_text(encoding="utf-8"))
            live_sc = art.get("oof_c_index")
            if live_sc is None:
                live_sc = art.get("oof_rmse")
            if isinstance(live_sc, int | float):
                sc_f = round(float(live_sc), 6)
                if len(traj_vals) == 1 and abs(sc_f - traj_vals[0]) > 1e-8:
                    traj_vals.append(sc_f)
                    lift_map = {"Pipeline": round(sc_f - traj_vals[0], 6)}
                if len(set(fold_vals)) <= 1:
                    fold_vals = [sc_f for _ in fold_vals]
        except (OSError, ValueError, TypeError, KeyError):
            pass
    return traj_vals, fold_vals, lift_map


def render_executive_governance_dashboard(
    model_dir: Path | None = None,
    output_path: str | Path = "executive_cv_governance.png",
) -> Path:
    """Render 4-panel Executive Validation & Attribution Dashboard (zero crowded x-ticks)."""
    from matplotlib.ticker import MaxNLocator

    out_file = Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    traj_vals, fold_vals, lift_map = _resolve_live_governance_series(model_dir)
    f_vals = np.asarray(fold_vals, dtype=np.float64)
    f_mean = float(np.mean(f_vals))
    f_sd = float(np.std(f_vals))
    fig, axes = plt.subplots(2, 2, figsize=(13.5, 7.8), facecolor="#F8FAFC")
    fig.suptitle(
        f"um-game-playing-strength-of-mcts-variants — Verified 5-Fold OOF Governance (RMSE = {f_mean:.5f})",
        fontsize=13,
        fontweight="bold",
        color="#0F172A",
        y=0.98,
    )
    for ax in axes.flat:
        ax.set_facecolor("#FFFFFF")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#CBD5E1")
        ax.spines["bottom"].set_color("#CBD5E1")
        ax.grid(True, linestyle="--", alpha=0.45, color="#CBD5E1")

    # Panel 1: Promoted Milestone Trajectory (integer x-ticks only)
    ax0 = axes[0, 0]
    xs = np.arange(len(traj_vals))
    ys = np.asarray(traj_vals, dtype=np.float64)
    ax0.plot(xs, ys, marker="o", linewidth=2.4, color="#0284C7", markersize=5.5)
    ax0.fill_between(xs, float(np.min(ys)), ys, color="#0284C7", alpha=0.10)
    ax0.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    ax0.set_title("1. Iterative Out-of-Fold Promotion Trajectory", fontweight="bold", fontsize=10.5)
    ax0.set_xlabel("Promoted Milestone Index")
    ax0.set_ylabel("RMSE")

    # Panel 2: Strict K-Fold Stability Band
    ax1 = axes[0, 1]
    f_idx = np.arange(1, len(f_vals) + 1)
    ax1.bar(f_idx, f_vals, width=0.56, color="#059669", alpha=0.88, edgecolor="#047857")
    ax1.axhline(
        f_mean, color="#0F172A", linestyle="--", linewidth=1.4, label=f"Mean {f_mean:.4f}"
    )
    ax1.axhspan(
        f_mean - f_sd, f_mean + f_sd, color="#059669", alpha=0.12, label=f"±1 SD ({f_sd:.4f})"
    )
    ax1.set_xticks(f_idx)
    y_pad = max(f_sd * 3.2, abs(f_mean) * 0.015, 1e-3)
    ax1.set_ylim(float(np.min(f_vals)) - y_pad, float(np.max(f_vals)) + y_pad)
    ax1.set_title("2. Strict K-Fold OOF Stability", fontweight="bold", fontsize=10.5)
    ax1.set_xlabel("Validation Fold")
    ax1.set_ylabel("RMSE")
    ax1.legend(loc="lower right", frameon=True, fontsize=8.5)

    # Panel 3: Top Signal Predictors (horizontal bars -> readable labels)
    ax2 = axes[1, 0]
    feat_labels = [c[:26] for c in (TOP_SIGNAL_PREVIEW or ["feature_0"])[:10]][::-1]
    ranks = np.linspace(0.38, 1.0, len(feat_labels))
    if model_dir is not None and (model_dir / "fold_0.lgb").is_file():
        try:
            import lightgbm as lgb

            lgb_txt = (model_dir / "fold_0.lgb").read_text(encoding="utf-8")
            bst = lgb.Booster(model_str=lgb_txt)
            gains = np.asarray(bst.feature_importance(importance_type="gain"), dtype=np.float64)
            if len(gains) >= len(feat_labels) and float(np.max(gains)) > 0:
                top_order = np.argsort(gains)[-len(feat_labels) :]
                names_raw = bst.feature_name()
                feat_labels = [
                    (
                        TOP_SIGNAL_PREVIEW[i]
                        if i < len(TOP_SIGNAL_PREVIEW) and names_raw[i].startswith("Column_")
                        else str(names_raw[i])
                    )[:26]
                    for i in top_order
                ]
                ranks = gains[top_order] / float(np.max(gains))
        except (OSError, ValueError, TypeError, KeyError):
            pass
    ax2.barh(feat_labels, ranks, color="#6366F1", alpha=0.90)
    ax2.set_title(
        "3. Top Predictor Gain Attribution (Normalized)", fontweight="bold", fontsize=10.5
    )
    ax2.set_xlabel("Relative Gain Importance")

    # Panel 4: Cumulative Lift by Topology Stage
    ax3 = axes[1, 1]
    st_names = list(lift_map.keys())[::-1]
    st_vals = [float(lift_map[k]) for k in st_names]
    st_colors = ["#059669" if v >= 0 else "#E11D48" for v in st_vals]
    ax3.barh(st_names, st_vals, color=st_colors, alpha=0.88)
    ax3.axvline(0.0, color="#64748B", linewidth=1.0)
    ax3.set_title("4. Verified OOF Lift by Pipeline Stage", fontweight="bold", fontsize=10.5)
    ax3.set_xlabel("Cumulative RMSE Delta")

    plt.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
    fig.savefig(out_file, dpi=150, bbox_inches="tight")
    plt.close(fig)
    _display_dashboard_inline(out_file)
    return out_file


def render_holdout_inference_diagnostics(
    fold_predictions: list[np.ndarray],
    ensemble_preds: np.ndarray,
    pred_col: str,
    output_path: str | Path = "holdout_inference_diagnostics.png",
) -> Path:
    """Render 3-panel Holdout Distribution, Fold Concordance & Epistemic Uncertainty chart."""
    out_file = Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.2), facecolor="#F8FAFC")
    fig.suptitle(
        f"K-Fold Ensemble Holdout Diagnostics (n = {len(ensemble_preds):,} scored rows)",
        fontsize=12.5,
        fontweight="bold",
        color="#0F172A",
        y=1.01,
    )
    for ax in axes:
        ax.set_facecolor("#FFFFFF")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(True, linestyle="--", alpha=0.40, color="#CBD5E1")

    # Panel 1: Holdout Ensemble Score Distribution + Quantiles
    n_bins = min(32, max(6, int(np.sqrt(max(len(ensemble_preds), 4))) * 2))
    axes[0].hist(ensemble_preds, bins=n_bins, color="#0284C7", alpha=0.85, edgecolor="white")
    if len(ensemble_preds) >= 3:
        p10, p50, p90 = np.percentile(ensemble_preds, [10.0, 50.0, 90.0])
        axes[0].axvline(p50, color="#0F172A", linestyle="--", linewidth=1.5, label=f"P50={p50:.3f}")
        axes[0].axvline(p10, color="#D97706", linestyle=":", linewidth=1.3, label=f"P10={p10:.3f}")
        axes[0].axvline(p90, color="#059669", linestyle=":", linewidth=1.3, label=f"P90={p90:.3f}")
        axes[0].legend(fontsize=8, frameon=True)
    axes[0].set_title("1. Ensemble Prediction Density", fontweight="bold", fontsize=10)
    axes[0].set_xlabel(pred_col)
    axes[0].set_ylabel("Row Count")

    # Panel 2: Cross-Fold Agreement (Fold 0 vs Remaining Fold Mean)
    mat = np.vstack(fold_predictions)
    f0 = mat[0]
    f_rest = np.mean(mat[1:], axis=0) if mat.shape[0] > 1 else f0
    axes[1].scatter(f0, f_rest, s=22, alpha=0.65, color="#059669", edgecolors="none")
    lo = float(min(np.min(f0), np.min(f_rest)))
    hi = float(max(np.max(f0), np.max(f_rest)))
    if hi > lo:
        axes[1].plot([lo, hi], [lo, hi], linestyle="--", color="#0F172A", linewidth=1.2)
    corr = (
        float(np.corrcoef(f0, f_rest)[0, 1])
        if len(f0) >= 3 and float(np.std(f0)) > 1e-9 and float(np.std(f_rest)) > 1e-9
        else 1.0
    )
    axes[1].set_title(f"2. Cross-Fold Concordance (r = {corr:.4f})", fontweight="bold", fontsize=10)
    axes[1].set_xlabel("Fold 0 Prediction")
    axes[1].set_ylabel("Fold 1..K Mean")

    # Panel 3: Epistemic Uncertainty (Per-Row Fold Std vs Score Quantile)
    row_std = np.std(mat, axis=0) if mat.shape[0] > 1 else np.zeros_like(ensemble_preds)
    order = np.argsort(ensemble_preds)
    n_groups = min(10, max(1, len(ensemble_preds)))
    splits = np.array_split(order, n_groups)
    bin_ids = np.arange(1, len(splits) + 1)
    bin_stds = [float(np.mean(row_std[idx])) if len(idx) > 0 else 0.0 for idx in splits]
    axes[2].bar(bin_ids, bin_stds, width=0.6, color="#6366F1", alpha=0.88)
    axes[2].set_xticks(bin_ids)
    axes[2].set_title("3. Epistemic Spread Across Score Bins", fontweight="bold", fontsize=10)
    axes[2].set_xlabel("Prediction Quantile Bin (Low → High)")
    axes[2].set_ylabel("Mean Cross-Fold Std")

    plt.tight_layout()
    fig.savefig(out_file, dpi=150, bbox_inches="tight")
    plt.close(fig)
    _display_dashboard_inline(out_file)
    return out_file


ARTIFACT_PATH = Path("artifacts/model_artifact.json")
TEST_PATH = Path("data/test.csv")
SAMPLE_SUB_PATH = Path("data/sample_submission.csv")
if ARTIFACT_PATH.is_file() and TEST_PATH.is_file():
    artifact = load_trained_artifact(ARTIFACT_PATH)
    test_df = pl.read_csv(TEST_PATH)
    sample_sub = pl.read_csv(SAMPLE_SUB_PATH) if SAMPLE_SUB_PATH.is_file() else None
    predictions = run_inference_pipeline(artifact, test_df)
    sub_df = format_submission_frame(
        test_df, predictions, sample_sub_df=sample_sub, target_col="utility_agent1"
    )
    sub_df.write_csv("submission.csv")
    render_holdout_inference_diagnostics(
        [predictions], predictions, sub_df.columns[-1], "artifacts/holdout_diagnostics.png"
    )
    print("Scored rows from saved artifact:", len(predictions), "-> submission.csv")
else:
    print("Run train.ipynb / pipeline.ipynb first to create artifacts/model_artifact.json.")
